In [3]:
# ===== INSTALAÇÕES (APENAS NO GOOGLE COLAB) =====
!apt-get update > /dev/null
!apt install chromium-chromedriver -y > /dev/null
!cp /usr/lib/chromium-browser/chromedriver /usr/bin
!pip install selenium pandas openpyxl > /dev/null

# ===== IMPORTS =====
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import os, re, requests, zipfile
import pandas as pd
from datetime import datetime
import time

# ===== CONFIGURAÇÃO DE PASTAS =====
data_hoje = datetime.now().strftime('%Y-%m-%d')
pasta_base = f"{data_hoje}_casarossi"
pasta_imagens = os.path.join(pasta_base, "imagens")
os.makedirs(pasta_imagens, exist_ok=True)

# ===== CONFIGURAÇÃO DO DRIVER =====
options = Options()
options.binary_location = "/usr/bin/chromium-browser"
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 10)

# ===== COLETAR LINKS DA LISTAGEM (LIMITADO A 10) =====
categoria_url = "https://www.casarossi.com.br/pisos-e-revestimentos/porcelanatos/acetinado"
print("🔎 Coletando até 10 links dos produtos da Casa Rossi...")

driver.get(categoria_url)
time.sleep(3)
soup = BeautifulSoup(driver.page_source, 'html.parser')

links = []
for item in soup.find_all('li', class_='item flex'):
    a = item.find('a', class_='space-image')
    if a:
        href = a['href']
        if not href.startswith("http"):
            href = "https://www.casarossi.com.br" + href
        if href not in links:
            links.append(href)
    if len(links) >= 10:
        break

print(f"🔗 {len(links)} links coletados.")

# ===== FUNÇÃO PARA EXTRAIR DETALHES COM SELENIUM =====
def extrair_detalhes_casarossi(driver, url):
    try:
        driver.get(url)
        time.sleep(3)
        soup = BeautifulSoup(driver.page_source, 'html.parser')

        # Nome
        try:
            nome = soup.find("h1", class_="fn").get_text(strip=True)
        except:
            nome = "N/A"

        # Preço por caixa
        try:
            preco_caixa = soup.find("span", class_="PrecoPrincipal").get_text(strip=True).replace("R$", "")
        except:
            preco_caixa = "N/A"

        # Preço por m²
        try:
            preco_m2 = soup.find("div", class_="preco-por-metro").get_text(strip=True)
        except:
            preco_m2 = "N/A"

        # m² por caixa
        try:
            m2_caixa = soup.find("div", class_="metro-na-caixa").get_text(strip=True)
        except:
            m2_caixa = "N/A"

        # Descrição
        try:
            descricao = ''
            for p in soup.find_all('p'):
                if 'Informações do Produto' in p.get_text():
                    descricao = p.get_text(separator='\n', strip=True)
                    break
        except:
            descricao = "N/A"

        # Imagem
        try:
            img_tag = soup.find("div", class_="zoom").find("img")
            imagem_url = img_tag["src"]
        except:
            imagem_url = "Não encontrado"

        # Código (placeholder com base no nome ou índice)
        codigo = re.sub(r'[\\/*?:"<>|]', "", nome)[:50] if nome != "Não encontrado" else "produto"

        return [nome, codigo, m2_caixa, descricao, preco_caixa, preco_caixa, preco_m2, imagem_url, url]

    except Exception as e:
        print(f"❌ Erro ao coletar dados de {url}: {e}")
        return None

# ===== FUNÇÃO PARA SALVAR IMAGEM =====
def salvar_imagem(url_img, nome_base):
    if url_img == "Não encontrado": return
    try:
        resposta = requests.get(url_img, timeout=10)
        if resposta.status_code == 200:
            caminho = os.path.join(pasta_imagens, f"{nome_base}.jpg")
            with open(caminho, 'wb') as f:
                f.write(resposta.content)
    except Exception as e:
        print(f"❌ Erro ao baixar imagem: {e}")

# ===== COLETAR DADOS DOS PRODUTOS =====
dados_produtos = []
for i, link in enumerate(links):
    print(f"[{i+1}/{len(links)}] Coletando dados de: {link}")
    resultado = extrair_detalhes_casarossi(driver, link)
    if resultado:
        dados_produtos.append(resultado)
        codigo = re.sub(r'[\\/*?:"<>|]', "", resultado[1])
        salvar_imagem(resultado[7], codigo)

driver.quit()

# ===== CRIAR DATAFRAME =====
df = pd.DataFrame(dados_produtos, columns=[
    "Nome", "Código", "Especificações", "Descrição", "Preço Original",
    "Preço Promocional", "Preço por m²", "Imagem", "URL"
])

# ===== LIMPAR PREÇOS =====
def limpar_preco(valor):
    try:
        return float(valor.replace("R$", "").replace(".", "").replace(",", ".").strip())
    except:
        return None

df["Preço Original"] = df["Preço Original"].apply(limpar_preco)
df["Preço Promocional"] = df["Preço Promocional"].apply(limpar_preco)
df["Preço por m²"] = df["Preço por m²"].apply(limpar_preco)

# ===== ADICIONAR COLUNA DE LOJA =====
df["Loja"] = "Casa Rossi"

# ===== REORDENAR COLUNAS =====
df = df[[
    "Código", "Nome", "Preço Promocional", "Preço Original", "Preço por m²",
    "Especificações", "Descrição", "Imagem", "URL", "Loja"
]]

# ===== SALVAR CSV E EXCEL =====
csv_path = os.path.join(pasta_base, f"casarossi_{data_hoje}.csv")
xlsx_path = os.path.join(pasta_base, f"casarossi_{data_hoje}.xlsx")
df.to_csv(csv_path, sep=';', index=False, encoding='utf-8-sig')
df.to_excel(xlsx_path, index=False)

print(f"\n📄 CSV salvo em: {csv_path}")
print(f"📊 Excel salvo em: {xlsx_path}")

# ===== COMPACTAR IMAGENS =====
zip_path = os.path.join(pasta_base, f"imagens_casarossi_{data_hoje}.zip")
with zipfile.ZipFile(zip_path, 'w') as zipf:
    for root, dirs, files in os.walk(pasta_imagens):
        for file in files:
            zipf.write(os.path.join(root, file), arcname=file)

print(f"📦 Imagens compactadas em: {zip_path}")


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


cp: '/usr/lib/chromium-browser/chromedriver' and '/usr/bin/chromedriver' are the same file
🔎 Coletando até 10 links dos produtos da Casa Rossi...
🔗 10 links coletados.
[1/10] Coletando dados de: https://www.casarossi.com.br/pisos-e-revestimentos/porcelanatos/acetinado/piso-formigres-66x66cm-acetinado-retificado-coimbra-2-18m
[2/10] Coletando dados de: https://www.casarossi.com.br/pisos-e-revestimentos/porcelanatos/acetinado/porcelanato-retificado-delta-72x72cm-out-itaara-raven-2-61m
[3/10] Coletando dados de: https://www.casarossi.com.br/pisos-e-revestimentos/porcelanatos/acetinado/porcelanatoincepa-90x90cm-acetinado-duo-concret-cris-2-40m
[4/10] Coletando dados de: https://www.casarossi.com.br/pisos-e-revestimentos/porcelanatos/acetinado/porcelanato-helena-62x121cm-acetinado-120133-mint-2-25m
[5/1